In [145]:
# ============================================================
# 🎮 VIDEO GAME SALES DASHBOARD
# ============================================================



In [159]:
import pandas as pd
import panel as pn
import plotly.express as px


In [161]:

pn.extension('plotly')

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("cleaned_vgsales.csv")

df["Total Sales"] = pd.to_numeric(df["Total Sales"], errors="coerce")
df["Critic Score"] = pd.to_numeric(df["Critic Score"], errors="coerce")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

df = df.dropna(subset=["Total Sales", "Year"]).head(2000)

# -------------------------
# FILTERS
# -------------------------
console_filter = pn.widgets.Select(
    name="Console",
    options=["All"] + sorted(df["Console"].dropna().unique())
)

genre_filter = pn.widgets.Select(
    name="Genre",
    options=["All"] + sorted(df["Genre"].dropna().unique())
)

# -------------------------
# FILTER FUNCTION
# -------------------------
def get_data():
    d = df.copy()

    if console_filter.value != "All":
        d = d[d["Console"] == console_filter.value]

    if genre_filter.value != "All":
        d = d[d["Genre"] == genre_filter.value]

    return d

# -------------------------
# STYLE (dark charts)
# -------------------------
def style(fig):
    fig.update_layout(
        plot_bgcolor="#111",
        paper_bgcolor="#111",
        font=dict(color="white"),
        margin=dict(l=80, r=40, t=60, b=80),
        xaxis=dict(tickangle=-30, automargin=True),
        yaxis=dict(automargin=True)
    )
    return fig

# -------------------------
# CARD (rounded)
# -------------------------
def card(component):
    return pn.Column(
        component,
        styles={
            "background": "#1b1b1b",
            "border-radius": "14px",
            "padding": "15px",
            "border": "1px solid #333",
            "margin-bottom": "20px"
        }
    )

# ============================================================
# KPI CARDS (LEFT SIDE)
# ============================================================

def kpi_cards(d):

    total_games = len(d)
    total_sales = round(d["Total Sales"].sum(), 2)
    avg_score = round(d["Critic Score"].mean(), 1)
    top_genre = d["Genre"].mode()[0] if len(d) > 0 else "N/A"

    def kpi(title, value):
        return pn.pane.Markdown(f"""
        <div style="
            background:#1b1b1b;
            border-radius:14px;
            padding:15px;
            border:1px solid #333;
            margin-bottom:15px;
            text-align:center;
        ">
            <h4 style="color:#aaa;">{title}</h4>
            <h2 style="color:#00e676;">{value}</h2>
        </div>
        """)

    return pn.Column(
        kpi("Total Games", total_games),
        kpi("Total Sales", total_sales),
        kpi("Avg Score", avg_score),
        kpi("Top Genre", top_genre),
        width=280
    )

# ============================================================
# CHARTS (8)
# ============================================================

def chart1(d):
    d = d.sort_values("Total Sales", ascending=False).head(10)
    fig = px.bar(d, x="Total Sales", y="Title", orientation="h", title="Top 10 Games")
    return card(pn.pane.Plotly(style(fig)))

def chart2(d):
    fig = px.pie(d, names="Genre", values="Total Sales", title="Genre Share")
    return card(pn.pane.Plotly(style(fig)))

def chart3(d):
    d = d.groupby("Year")["Total Sales"].sum().reset_index()
    fig = px.line(d, x="Year", y="Total Sales", title="Sales Trend")
    return card(pn.pane.Plotly(style(fig)))

def chart4(d):
    r = d[["NA Sales","Japan Sales","PAL Sales","Other Sales"]].sum().reset_index()
    r.columns = ["Region","Sales"]
    fig = px.bar(r, x="Region", y="Sales", title="Regional Sales")
    return card(pn.pane.Plotly(style(fig)))

def chart5(d):
    d = d.dropna(subset=["Critic Score","Total Sales"]).head(1000)
    fig = px.scatter(d, x="Critic Score", y="Total Sales",
                     size="Total Sales", color="Genre",
                     title="Critic Score vs Sales")
    return card(pn.pane.Plotly(style(fig)))

def chart6(d):
    d = d.groupby("Console")["Total Sales"].sum().reset_index()
    fig = px.bar(d, x="Console", y="Total Sales", title="Console Performance")
    return card(pn.pane.Plotly(style(fig)))

def chart7(d):
    fig = px.box(d, x="Genre", y="Total Sales", title="Sales Distribution")
    return card(pn.pane.Plotly(style(fig)))

def chart8(d):
    fig = px.histogram(d, x="Critic Score", title="Critic Score Distribution")
    return card(pn.pane.Plotly(style(fig)))

# ============================================================
# BUILD DASHBOARD
# ============================================================

def build_dashboard():
    d = get_data()

    # LEFT KPI
    kpi_col = kpi_cards(d)

    # RIGHT CHARTS (ONE COLUMN)
    overview_charts = pn.Column(
        chart1(d),
        chart2(d),
        chart3(d),
        chart4(d),
        sizing_mode="stretch_width"
    )

    analysis_charts = pn.Column(
        chart5(d),
        chart6(d),
        chart7(d),
        chart8(d),
        sizing_mode="stretch_width"
    )

    tab1 = pn.Row(kpi_col, overview_charts)
    tab2 = pn.Row(kpi_col, analysis_charts)

    return pn.Tabs(
        ("Overview", tab1),
        ("Analysis", tab2)
    )

# ============================================================
# MAIN DASHBOARD
# ============================================================

dashboard = pn.Column(
    pn.pane.Markdown("## 🎮 Video Game Sales Dashboard", styles={"color": "white"}),

    pn.Row(console_filter, genre_filter),

    pn.bind(lambda: build_dashboard())
)


In [162]:
pn.serve(dashboard, show=True)

Launching server at http://localhost:62466


2026-04-23 19:22:57.525 200 GET / (127.0.0.1) 235.41ms
2026-04-23 19:22:57.610 200 GET /static/extensions/panel/bundled/plotlyplot/maplibre-gl@4.4.1/dist/maplibre-gl.css?v=1.8.10 (127.0.0.1) 2.11ms
2026-04-23 19:22:57.616 200 GET /static/js/bokeh-tables.min.js?v=62532f6ff926a41c47def08668ee2cec06b351a8e8a5ca01a21fdc78811c3fc5d6b048deaf949f6abc5c6963b7bbf56ea285d6ada0b7a61191449d0d115bf737 (127.0.0.1) 4.36ms
2026-04-23 19:22:57.619 200 GET /static/extensions/panel/bundled/reactiveesm/es-module-shims@%5E1.10.0/dist/es-module-shims.min.js (127.0.0.1) 6.93ms
2026-04-23 19:22:57.622 200 GET /static/js/bokeh-gl.min.js?v=b803cb051212c1e9f0fba71c62739c0cfb38114322d67a2ace60eee732710b6541d80907d9fcf0afb2f962c368f1db1e30b8800652d9510d1b801727f9669eb2 (127.0.0.1) 9.60ms
2026-04-23 19:22:57.633 200 GET /static/extensions/panel/panel.min.js?v=b081f23dfb5efd1bbe84e7fda7427a25cfc354ce9fbf1fd8e5c35899acb2b3bb (127.0.0.1) 19.59ms
2026-04-23 19:22:57.638 200 GET /static/js/bokeh-widgets.min.js?v=aa4c33f